# Week 4, Day 2 — LangGraph, the Orchestration Layer
### Local Models Edition — by Abhishek

Yesterday you met the building blocks. Today LangGraph wires those blocks into
a graph it runs for you, keeping track of state and memory as it goes.

**No paid API needed.** Search uses DuckDuckGo (free, no key). Notifications
default to a local print/log — plug in real Pushover keys if you have them.


## What LangGraph is

LangGraph lets you describe work as a graph: nodes (Python functions) joined by
edges that decide what runs next. A shared **state** object passes from node to
node, and a snapshot can be saved at every step so your app can remember and
recover. LangGraph is agnostic to what it orchestrates — you don't even need an
LLM in the picture.


## 0. Setup

In [ ]:
import importlib.util
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BACKEND = "huggingface" if IN_COLAB else "ollama"
print(f"Backend: {BACKEND}")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q langgraph langchain-core langchain-huggingface transformers torch accelerate duckduckgo-search gradio
else:
    %pip install -q langgraph langchain-core langchain-ollama ollama duckduckgo-search gradio


In [ ]:
import random
from typing import Annotated
from typing_extensions import TypedDict
from IPython.display import Image, display
import gradio as gr

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.tools import tool

if BACKEND == "huggingface":
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
    pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", max_new_tokens=250)
    llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))
else:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2:3b", temperature=0.3)

print("llm ready")


## The State, and a word about reducers

The state flows through the graph, described with a `TypedDict`. By default a
new value replaces the old one. A **reducer** says how to combine old and new
instead — LangGraph ships `add_messages`, which appends rather than overwrites.


In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


## Five steps to a graph, with no LLM in sight

define state → start a builder → add nodes → add edges → compile.


In [ ]:
nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Eels", "Pickles"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "sparkly", "haunted"]

def silly_node(state: State) -> dict:
    sentence = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    return {"messages": [{"role": "assistant", "content": sentence}]}

builder = StateGraph(State)
builder.add_node("silly", silly_node)
builder.add_edge(START, "silly")
builder.add_edge("silly", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "say something"}]})
print(result["messages"][-1].content)


Now swap the silly node for one that calls a local model.

In [ ]:
def chatbot_node(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "What is a directed graph, in one sentence?"}]})
print(result["messages"][-1].content)


## Adding tools: a node, a condition, and a loop

`ToolNode` runs whatever tools the model asked for. `tools_condition` routes to
the tools node when the model wants a tool, and to `END` otherwise.

Two free, no-key tools: **DuckDuckGo search** (via `duckduckgo-search`, no API
key at all) and a **local notification** tool that logs to a file — swap in
real Pushover credentials if you have them, using the commented-out block.


In [ ]:
from duckduckgo_search import DDGS

@tool
def web_search(query: str) -> str:
    """Search the web and return a short summary of the top results."""
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=3))
    if not results:
        return "No results found."
    return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)


@tool
def send_notification(text: str) -> str:
    """Send a short notification to the user. Logs locally by default."""
    # --- Uncomment to send a REAL push notification via Pushover (free tier) ---
    # import os, requests
    # requests.post("https://api.pushover.net/1/messages.json", data={
    #     "token": os.getenv("PUSHOVER_TOKEN"), "user": os.getenv("PUSHOVER_USER"), "message": text,
    # })
    with open("notifications.log", "a") as f:
        f.write(text + "\n")
    print(f"[notification logged] {text}")
    return "Notification sent"

tools = [web_search, send_notification]
llm_with_tools = llm.bind_tools(tools)


In [ ]:
def chatbot_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content":
    "Use your search tool to tell me the ingredients in banoffee pie and send me a notification."}]})
print(result["messages"][-1].content)


## A second LLM node: the translator

Nodes don't have to be part of the tool loop. Any function can be a node,
including one that makes its own LLM call.


In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    spanish: str

def translator_node(state: State) -> dict:
    last = state["messages"][-1].content
    prompt = f"Translate this into Spanish, replying with the translation only:\n\n{last}"
    return {"spanish": llm.invoke(prompt).content}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END: "translator"})
builder.add_edge("tools", "chatbot")
builder.add_edge("translator", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "In one sentence, what is special about a banana?"}]})
print(result["messages"][-1].content)
print(result["spanish"])


## Observability with LangSmith

LangSmith traces every model and tool call, **including local Ollama/Hugging
Face calls** — it's provider-agnostic, same as the graph itself. Sign up free
at [smith.langchain.com](https://smith.langchain.com), then add to your `.env`:

```
LANGSMITH_TRACING=true
LANGSMITH_ENDPOINT=https://api.smith.langchain.com
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_PROJECT=agentic-track-local
```

Every LangChain/LangGraph run is then traced automatically — worth turning on
for your own work, even (especially) while debugging local-model tool use.


## Memory: why the graph forgets, and how to fix it

One call to `invoke` is one run of the graph. The reducer accumulates messages
*within* that run, but the next `invoke` starts fresh. A **checkpointer** saves
a snapshot after every super-step, filed under a `thread_id`.


In [ ]:
memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END: "translator"})
builder.add_edge("translator", END)
builder.add_edge("tools", "chatbot")
graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "conversation-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Abhishek."}]}, config)
second = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)
print(second["messages"][-1].content)
print(second["spanish"])


## Persisting to SQLite

`MemorySaver` vanishes when the program stops. The SQLite checkpointer writes
to a database file so the same graph code survives restarts.


In [ ]:
with SqliteSaver.from_conn_string("memory.db") as sql_memory:
    durable_graph = builder.compile(checkpointer=sql_memory)
    sql_config = {"configurable": {"thread_id": "conversation-2"}}
    durable_graph.invoke({"messages": [{"role": "user", "content": "Remember that my favorite color is orange."}]}, sql_config)
    reply = durable_graph.invoke({"messages": [{"role": "user", "content": "What is my favorite color?"}]}, sql_config)
    print(reply["messages"][-1].content)
    print(reply["spanish"])


## Looking inside the state, and travelling back in time

Because the checkpointer saves a snapshot at every super-step, you can inspect
the current state and walk the whole history, and resume from any earlier point.


In [ ]:
snapshot = graph.get_state(config)
messages = snapshot.values["messages"]
print("Messages stored so far:", len(messages))
for message in messages:
    print(message.content)

history = list(graph.get_state_history(config))
print("Number of saved checkpoints:", len(history))


In [ ]:
for h in reversed(history):
    m = h.metadata
    queued = ", ".join(t.name for t in h.tasks) or "(run complete)"
    print(f"step {m['step']:>2}  {m['source']:<5}  messages={len(h.values.get('messages', []))}  about to run: {queued}")


In [ ]:
earlier = history[len(history) // 2]
print("A checkpoint from earlier held", len(earlier.values["messages"]), "messages")

replay_config = {"configurable": {"thread_id": "conversation-1",
                                  "checkpoint_id": earlier.config["configurable"]["checkpoint_id"]}}
resumed = graph.invoke(None, replay_config)
print("Resumed from the past; the graph now holds", len(resumed["messages"]), "messages")

again = graph.invoke({"messages": [{"role": "user", "content": "What do you know about me?"}]}, replay_config)
print(again["messages"][-1].content)
print(again["spanish"])


## Adding a UI

In [ ]:
def chat(user_input: str, history):
    config = {"configurable": {"thread_id": "gradio-session4"}}
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config)
    return f"{result['messages'][-1].content}\n\n*{result['spanish']}*"

gr.ChatInterface(chat).launch()


## Recap, and where we are heading

You built graphs by hand: state with a reducer, nodes, edges, a conditional
edge, a tool loop, a second LLM node, memory in two flavors, and a way to
inspect and replay the whole thing — all on a free local model.

Tomorrow, Layer 3: `create_agent` builds a graph exactly like this tool loop
in a single line.

## Exercise
Add a third tool to the graph, then ask a question that needs a search
followed by a notification and uses your tool, so the tool loop runs more
than once. If you turned on LangSmith, trace exactly which nodes ran.
